# Closed-loop simulation

Evaluate deterministic controllers with perfect simulator observations.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Scenario and driver

Perception predictions are not connected to control in this milestone.

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt

from autonomous_vehicles.control.driver import Driver
from autonomous_vehicles.evaluation.driving import driving_metrics
from autonomous_vehicles.simulation.scenarios import SCENARIOS, make_scenario
from autonomous_vehicles.simulation.world import run_episode
from autonomous_vehicles.visualization import plot_episode, plot_scene

print(list(SCENARIOS))
scenario = make_scenario("lane_following")

## Run an episode

The simulator integrates bounded controls and checks collision using oriented rectangles.

In [ ]:
episode = run_episode(scenario, Driver("pursuit"))
metrics = driving_metrics(episode, scenario)
print(metrics.to_dict())
plot_episode(episode, scenario)
plt.show()
assert not episode.collided

## Compare lateral controllers

Use the identical initial state and road for each controller.

In [ ]:
results = {}
for controller in ("pursuit", "stanley"):
    result = run_episode(scenario, Driver(controller))
    results[controller] = driving_metrics(result, scenario)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(results.keys(), [metric.lateral_rmse_meters for metric in results.values()])
ax.set_ylabel("Lateral RMSE (m)")
plt.show()

## Stopped obstacle

The stopping envelope uses actor positions to limit desired speed.

In [ ]:
stopped = make_scenario("stopped_vehicle")
safe_episode = run_episode(stopped, Driver(safety=True))
unsafe_episode = run_episode(stopped, Driver(safety=False))
print("Envelope:", safe_episode.termination, safe_episode.final.speed)
print("Disabled:", unsafe_episode.termination, unsafe_episode.final.speed)
plot_episode(safe_episode, stopped)
plt.show()
assert not safe_episode.collided

## Observe the crossing scenario

The current corridor rule can miss actors before they enter the corridor. Treat failures as research evidence.

In [ ]:
crossing = make_scenario("crossing_pedestrian")
crossing_episode = run_episode(crossing)
print(driving_metrics(crossing_episode, crossing).to_dict())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_scene(crossing, crossing.initial, ax=axes[0])
plot_scene(crossing, crossing_episode.final, ax=axes[1])
plt.tight_layout()
plt.show()

## Time-step sensitivity

Discrete collision checks can miss fast motion. Internal substeps reduce, but do not eliminate, this limitation.

In [ ]:
for dt in (0.05, 0.1, 0.2):
    candidate = replace(scenario, dt=dt, duration=5)
    result = run_episode(candidate)
    measured = driving_metrics(result, candidate)
    print(
        {
            "dt": dt,
            "termination": result.termination,
            "lateral_rmse_m": measured.lateral_rmse_meters,
            "progress_m": measured.progress_meters,
        }
    )

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.